# Resume RAG: experimentation and analysis
Run this notebook from the project root after indexing resumes with `python resume_rag.py`.

In [4]:
from pathlib import Path
import json, time
import pandas as pd
from job_matcher import match_job

ROOT = Path.cwd()
if not (ROOT / 'job_matcher.py').exists():
    ROOT = ROOT.parent
    import os; os.chdir(ROOT)
print('Project root:', ROOT)

Project root: c:\Users\HP\Desktop\AIRTRIBE\RAG project\resume_rag_job_matcher


In [3]:
import chromadb
import pandas as pd
from pathlib import Path
import sys

ROOT = Path(
    r"C:\Users\HP\Desktop\AIRTRIBE\RAG project\resume_rag_job_matcher"
)
sys.path.insert(0, str(ROOT))

from job_matcher import match_job

print("All imports successful!")
print("Project:", ROOT)
print("Python:", sys.executable)

All imports successful!
Project: C:\Users\HP\Desktop\AIRTRIBE\RAG project\resume_rag_job_matcher
Python: c:\Users\HP\AppData\Local\Programs\Python\Python311\python.exe


In [5]:
jd_path = Path('job_descriptions/data_scientist.txt')
jd = jd_path.read_text(encoding='utf-8')
result = match_job(jd, top_k=10, enforce_must_haves=False)
print(json.dumps(result, indent=2)[:5000])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

{
  "job_description": "Data Scientist\nWe are looking for a Data Scientist with 5+ years of experience. Must have Python, SQL, machine learning, statistics, Pandas, and Scikit-learn. Experience building predictive models, running experiments, and communicating findings to product stakeholders is important. Cloud deployment experience is a plus.\n",
  "filters": {
    "minimum_experience_years_detected": 5,
    "must_have_skills_detected": [
      "Python",
      "SQL",
      "Machine Learning",
      "Scikit-learn",
      "Pandas",
      "Statistics"
    ]
  },
  "top_matches": [
    {
      "candidate_name": "Aarav Sharma",
      "resume_path": "resumes/candidate_01.txt",
      "match_score": 70,
      "matched_skills": [
        "Machine Learning",
        "Pandas",
        "Python",
        "SQL",
        "Scikit-learn",
        "Statistics"
      ],
      "relevant_excerpts": [
        "5 years of professional experience in data scientist responsibilities. Collaborated with cross-

In [6]:
rows = []
for jd_file in sorted(Path('job_descriptions').glob('*.txt')):
    jd_text = jd_file.read_text(encoding='utf-8')
    out = match_job(jd_text, top_k=10, enforce_must_haves=False)
    rows.append({'job_description': jd_file.name,
                 'latency_ms': out['metrics']['latency_ms'],
                 'results_returned': out['metrics']['results_returned'],
                 'top_candidate': out['top_matches'][0]['candidate_name'] if out['top_matches'] else None,
                 'top_score': out['top_matches'][0]['match_score'] if out['top_matches'] else None})
metrics_df = pd.DataFrame(rows)
metrics_df

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

,job_description,latency_ms,results_returned,top_candidate,top_score
0,bi_analyst.txt,4372.87,10,Priya Reddy,69
1,data_engineer.txt,4640.62,10,Fatima Khan,72
2,data_scientist.txt,4575.42,10,Aarav Sharma,70
3,ml_engineer.txt,4264.67,10,Maya Patel,61
4,rag_engineer.txt,4663.56,10,Arjun Rao,67


## How to evaluate retrieval accuracy
This starter notebook measures latency and inspects results. For true precision@K/recall@K, manually label relevant resumes for each job description (ground truth), then compare retrieved candidates with those labels. Do not call an unlabelled similarity score 'accuracy'.

In [7]:
# Example template: replace these candidate filenames with manually judged relevant resumes.
ground_truth = {
    'data_scientist.txt': {'candidate_01.txt', 'candidate_15.txt', 'candidate_27.txt'},
    'rag_engineer.txt': {'candidate_05.txt', 'candidate_21.txt'},
}

def precision_at_k(retrieved_paths, relevant_paths, k=10):
    retrieved = retrieved_paths[:k]
    return sum(Path(p).name in relevant_paths for p in retrieved) / max(1, len(retrieved))

print('Fill in/verify ground_truth with human-reviewed labels before reporting evaluation metrics.')

Fill in/verify ground_truth with human-reviewed labels before reporting evaluation metrics.


In [8]:

from pathlib import Path
import json

print("Project folder:", Path.cwd())
print("Result files found:")

for file in Path.cwd().glob("*results*.json"):
    print("-", file.name)


Project folder: c:\Users\HP\Desktop\AIRTRIBE\RAG project\resume_rag_job_matcher
Result files found:
- bi_results.json
- data_engineer_results.json
- ml_results.json
- rag_results.json
- results.json


In [9]:

import json
from pathlib import Path

result_files = {
    "Data Scientist": "results.json",
    "RAG Engineer": "rag_results.json",
    "Data Engineer": "data_engineer_results.json",
    "BI Analyst": "bi_results.json",
    "ML Engineer": "ml_results.json",
}

for job_title, filename in result_files.items():
    path = Path(filename)

    print("\n" + "=" * 60)
    print(f"JOB: {job_title}")
    print("=" * 60)

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print("JSON structure:", type(data).__name__)

    if isinstance(data, dict):
        print("Available keys:", list(data.keys()))
    elif isinstance(data, list):
        print("Number of results:", len(data))

    print("Sample:")
    print(json.dumps(data, indent=2)[:1500])



JOB: Data Scientist
JSON structure: dict
Available keys: ['job_description', 'filters', 'top_matches', 'metrics', 'scoring_note']
Sample:
{
  "job_description": "Data Scientist\nWe are looking for a Data Scientist with 5+ years of experience. Must have Python, SQL, machine learning, statistics, Pandas, and Scikit-learn. Experience building predictive models, running experiments, and communicating findings to product stakeholders is important. Cloud deployment experience is a plus.\n",
  "filters": {
    "minimum_experience_years_detected": 5,
    "must_have_skills_detected": [
      "Python",
      "SQL",
      "Machine Learning",
      "Scikit-learn",
      "Pandas",
      "Statistics"
    ]
  },
  "top_matches": [
    {
      "candidate_name": "Aarav Sharma",
      "resume_path": "resumes/candidate_01.txt",
      "match_score": 70,
      "matched_skills": [
        "Machine Learning",
        "Pandas",
        "Python",
        "SQL",
        "Scikit-learn",
        "Statistics"
   

In [10]:

import json
import pandas as pd
from pathlib import Path

# Load the Data Scientist results
with open("results.json", "r", encoding="utf-8") as f:
    data = json.load(f)

matches = data.get("top_matches", [])

print("Job:", data.get("job_description", "").split("\n")[0])
print("Number of candidates:", len(matches))
print("\nAvailable candidate fields:")
if matches:
    print(list(matches[0].keys()))

# Display useful fields that actually exist in the JSON
preferred_columns = [
    "candidate_name",
    "resume_path",
    "match_score",
    "matched_skills",
    "reasoning",
]

columns = [col for col in preferred_columns if col in pd.DataFrame(matches).columns]

df = pd.DataFrame(matches)
display(df[columns].head(10))


Job: Data Scientist
Number of candidates: 10

Available candidate fields:
['candidate_name', 'resume_path', 'match_score', 'matched_skills', 'relevant_excerpts', 'reasoning', 'experience_years', 'missing_detected_skills', 'must_have_pass']


,candidate_name,resume_path,match_score,matched_skills,reasoning
0,Aarav Sharma,resumes/candidate_01.txt,70,"[Machine Learning, Pandas, Python, SQL, Scikit...",Semantic relevance: 50%. Matched 6 of 6 detect...
1,Emma Wilson,resumes/candidate_15.txt,63,"[Machine Learning, Pandas, Python, SQL, Statis...",Semantic relevance: 47%. Matched 5 of 6 detect...
2,Samuel Clark,resumes/candidate_25.txt,55,"[Machine Learning, Python, SQL]",Semantic relevance: 51%. Matched 3 of 6 detect...
3,Fatima Khan,resumes/candidate_07.txt,49,"[Python, SQL]",Semantic relevance: 48%. Matched 2 of 6 detect...
4,Divya Menon,resumes/candidate_28.txt,49,"[Python, SQL]",Semantic relevance: 48%. Matched 2 of 6 detect...
5,Ananya Iyer,resumes/candidate_09.txt,49,"[Machine Learning, Python, Statistics]",Semantic relevance: 40%. Matched 3 of 6 detect...
6,Neha Kulkarni,resumes/candidate_24.txt,49,"[Python, SQL, Statistics]",Semantic relevance: 40%. Matched 3 of 6 detect...
7,Benjamin Davis,resumes/candidate_17.txt,48,"[Python, SQL]",Semantic relevance: 46%. Matched 2 of 6 detect...
8,Sofia Garcia,resumes/candidate_11.txt,45,"[Python, SQL]",Semantic relevance: 42%. Matched 2 of 6 detect...
9,Daniel Brown,resumes/candidate_10.txt,42,[SQL],Semantic relevance: 45%. Matched 1 of 6 detect...


In [11]:

from pathlib import Path
import json
import pandas as pd

with open("results.json", "r", encoding="utf-8") as f:
    data = json.load(f)

matches = data["top_matches"]

def shorten(value, limit=250):
    if isinstance(value, list):
        value = "; ".join(str(x) for x in value[:2])
    elif value is None:
        value = ""
    else:
        value = str(value)
    return value if len(value) <= limit else value[:limit] + "..."

review = pd.DataFrame([
    {
        "candidate": m.get("candidate_name"),
        "score": m.get("match_score"),
        "experience_years": m.get("experience_years"),
        "matched_skills": shorten(m.get("matched_skills"), 180),
        "missing_skills": shorten(m.get("missing_detected_skills"), 180),
        "must_have_pass": m.get("must_have_pass"),
        "resume_evidence": shorten(m.get("relevant_excerpts"), 250),
    }
    for m in matches
])

with pd.option_context(
    "display.max_colwidth", 100,
    "display.width", 220
):
    display(review)


,candidate,score,experience_years,matched_skills,missing_skills,must_have_pass,resume_evidence
0,Aarav Sharma,70,5,Machine Learning; Pandas,,True,5 years of professional experience in data scientist responsibilities. Collaborated with cross-f...
1,Emma Wilson,63,6,Machine Learning; Pandas,Scikit-learn,False,"Python, SQL, Machine Learning, A/B Testing, Statistics, Pandas; 6 years of professional experien..."
2,Samuel Clark,55,5,Machine Learning; Python,Pandas; Scikit-learn,False,"Python, SQL, Spark, Machine Learning, AWS, Airflow; 5 years of professional experience in resear..."
3,Fatima Khan,49,6,Python; SQL,Machine Learning; Pandas,False,6 years of professional experience in data engineer responsibilities. Collaborated with cross-fu...
4,Divya Menon,49,8,Python; SQL,Machine Learning; Pandas,False,"SQL, Python, PostgreSQL, AWS, Linux, Data Governance"
5,Ananya Iyer,49,8,Machine Learning; Python,Pandas; SQL,False,"Python, Machine Learning, Deep Learning, PyTorch, Statistics, NLP; Ph.D. in Computer Science, In..."
6,Neha Kulkarni,49,5,Python; SQL,Machine Learning; Pandas,False,5 years of professional experience in financial data analyst responsibilities. Collaborated with...
7,Benjamin Davis,48,10,Python; SQL,Machine Learning; Pandas,False,"SQL, Python, Spark, AWS, Databricks, ETL, Data Governance; 10 years of professional experience i..."
8,Sofia Garcia,45,5,Python; SQL,Machine Learning; Pandas,False,"SQL, Power BI, Tableau, Excel, Python, Data Analysis; 5 years of professional experience in busi..."
9,Daniel Brown,42,5,SQL,Machine Learning; Pandas,False,"B.S. in Computer Science, University of Texas at Austin"


In [12]:

import json
from pathlib import Path

with open("results.json", "r", encoding="utf-8") as f:
    data = json.load(f)

for candidate in data["top_matches"][:3]:
    print("\n" + "=" * 60)
    print("CANDIDATE:", candidate.get("candidate_name"))
    print("=" * 60)

    for field in [
        "match_score",
        "experience_years",
        "matched_skills",
        "missing_detected_skills",
        "must_have_pass",
        "relevant_excerpts",
        "reasoning",
    ]:
        print(f"\n{field}:")
        print(json.dumps(candidate.get(field), indent=2, ensure_ascii=False))



CANDIDATE: Aarav Sharma

match_score:
70

experience_years:
5

matched_skills:
[
  "Machine Learning",
  "Pandas",
  "Python",
  "SQL",
  "Scikit-learn",
  "Statistics"
]

missing_detected_skills:
[]

must_have_pass:
true

relevant_excerpts:
[
  "5 years of professional experience in data scientist responsibilities. Collaborated with cross-functional teams, documented technical decisions, and delivered measurable improvements to internal workflows.",
  "Python, SQL, Machine Learning, Pandas, NumPy, Scikit-learn, Statistics",
  "Aarav Sharma\nData Scientist"
]

reasoning:
"Semantic relevance: 50%. Matched 6 of 6 detected job skills. Experience extracted: 5 years; requirement detected: 5 years. Evidence of matching skills: Machine Learning, Pandas, Python, SQL, Scikit-learn, Statistics. Relevant sections: Experience, Skills, General, Education."

CANDIDATE: Emma Wilson

match_score:
63

experience_years:
6

matched_skills:
[
  "Machine Learning",
  "Pandas",
  "Python",
  "SQL",
  "Stat

In [13]:

import json

with open("results.json", "r", encoding="utf-8") as f:
    data = json.load(f)

for name in ["Aarav Sharma", "Emma Wilson"]:
    candidate = next(
        (c for c in data["top_matches"]
         if c.get("candidate_name") == name),
        None
    )

    print("\n" + "=" * 45)
    print("CANDIDATE:", name)
    print("=" * 45)

    if candidate is None:
        print("Candidate not found")
        continue

    for field in [
        "match_score",
        "experience_years",
        "matched_skills",
        "missing_detected_skills",
        "must_have_pass",
        "reasoning",
    ]:
        print(f"{field}: {candidate.get(field)}")



CANDIDATE: Aarav Sharma
match_score: 70
experience_years: 5
matched_skills: ['Machine Learning', 'Pandas', 'Python', 'SQL', 'Scikit-learn', 'Statistics']
missing_detected_skills: []
must_have_pass: True
reasoning: Semantic relevance: 50%. Matched 6 of 6 detected job skills. Experience extracted: 5 years; requirement detected: 5 years. Evidence of matching skills: Machine Learning, Pandas, Python, SQL, Scikit-learn, Statistics. Relevant sections: Experience, Skills, General, Education.

CANDIDATE: Emma Wilson
match_score: 63
experience_years: 6
matched_skills: ['Machine Learning', 'Pandas', 'Python', 'SQL', 'Statistics']
missing_detected_skills: ['Scikit-learn']
must_have_pass: False
reasoning: Semantic relevance: 47%. Matched 5 of 6 detected job skills. Experience extracted: 6 years; requirement detected: 5 years. Evidence of matching skills: Machine Learning, Pandas, Python, SQL, Statistics. Job skills not detected in this resume: Scikit-learn. Relevant sections: Skills, Experience, 

In [14]:

from pathlib import Path
import json

with open("results.json", "r", encoding="utf-8") as f:
    data = json.load(f)

for name in ["Aarav Sharma", "Emma Wilson"]:
    candidate = next(
        c for c in data["top_matches"]
        if c["candidate_name"] == name
    )

    resume_path = Path(candidate["resume_path"])

    print("\n" + "=" * 60)
    print("RESUME:", name)
    print("=" * 60)

    if resume_path.exists():
        print(resume_path.read_text(encoding="utf-8")[:3000])
    else:
        print("Resume file not found:", resume_path.resolve())



RESUME: Aarav Sharma
Aarav Sharma
Data Scientist

SUMMARY
Built churn prediction models and customer segmentation pipelines. Deployed batch inference and presented model findings to product teams.

SKILLS
Python, SQL, Machine Learning, Pandas, NumPy, Scikit-learn, Statistics

EXPERIENCE
5 years of professional experience in data scientist responsibilities. Collaborated with cross-functional teams, documented technical decisions, and delivered measurable improvements to internal workflows.

EDUCATION
M.Tech in Data Science, National Institute of Technology

PROJECTS
Completed practical projects involving Python, stakeholder communication, version control, and reproducible documentation.


RESUME: Emma Wilson
Emma Wilson
Product Data Scientist

SUMMARY
Designed experiments, measured product outcomes, and developed propensity and retention models.

SKILLS
Python, SQL, Machine Learning, A/B Testing, Statistics, Pandas

EXPERIENCE
6 years of professional experience in product data scientis

In [16]:

from pathlib import Path

emma_path = Path("resumes/candidate_15.txt")

print("Resume path:", emma_path.resolve())

if emma_path.exists():
    print(emma_path.read_text(encoding="utf-8"))
else:
    print("Resume not found. Check the project folder.")


Resume path: C:\Users\HP\Desktop\AIRTRIBE\RAG project\resume_rag_job_matcher\resumes\candidate_15.txt
Emma Wilson
Product Data Scientist

SUMMARY
Designed experiments, measured product outcomes, and developed propensity and retention models.

SKILLS
Python, SQL, Machine Learning, A/B Testing, Statistics, Pandas

EXPERIENCE
6 years of professional experience in product data scientist responsibilities. Collaborated with cross-functional teams, documented technical decisions, and delivered measurable improvements to internal workflows.

EDUCATION
M.S. in Statistics, University of Michigan

PROJECTS
Completed practical projects involving Python, stakeholder communication, version control, and reproducible documentation.



In [17]:

from pathlib import Path

ground_truth = {
    "results.json": {
        "candidate_01.txt",  # Aarav Sharma: reviewed
        "candidate_15.txt",  # Emma Wilson: reviewed
    }
}

print("Initial reviewed labels:")
for job, candidates in ground_truth.items():
    print(job, sorted(candidates))


Initial reviewed labels:
results.json ['candidate_01.txt', 'candidate_15.txt']


In [18]:

from pathlib import Path

samuel_path = Path("resumes/candidate_25.txt")

print(samuel_path.read_text(encoding="utf-8"))


Samuel Clark
Research Data Engineer

SUMMARY
Prepared research datasets, implemented reproducible ETL, and supported model experiments.

SKILLS
Python, SQL, Spark, Machine Learning, AWS, Airflow

EXPERIENCE
5 years of professional experience in research data engineer responsibilities. Collaborated with cross-functional teams, documented technical decisions, and delivered measurable improvements to internal workflows.

EDUCATION
M.S. in Data Science, University of Wisconsin

PROJECTS
Completed practical projects involving Python, stakeholder communication, version control, and reproducible documentation.



In [19]:

from pathlib import Path

fatima_path = Path("resumes/candidate_07.txt")

print("File exists:", fatima_path.exists())
print("File size:", fatima_path.stat().st_size, "bytes")

text = fatima_path.read_text(encoding="utf-8")
print("Character count:", len(text))
print("Full resume below:\n")
print(repr(text))


File exists: True
File size: 605 bytes
Character count: 605
Full resume below:

'Fatima Khan\nData Engineer\n\nSUMMARY\nBuilt scalable ETL pipelines, improved data quality checks, and orchestrated scheduled data workflows.\n\nSKILLS\nPython, SQL, Spark, Airflow, ETL, Databricks, AWS\n\nEXPERIENCE\n6 years of professional experience in data engineer responsibilities. Collaborated with cross-functional teams, documented technical decisions, and delivered measurable improvements to internal workflows.\n\nEDUCATION\nM.S. in Data Engineering, University of Houston\n\nPROJECTS\nCompleted practical projects involving Python, stakeholder communication, version control, and reproducible documentation.\n'


In [20]:

from pathlib import Path

divya_path = Path("resumes/candidate_28.txt")
print(divya_path.read_text(encoding="utf-8"))


Divya Menon
Database Administrator

SUMMARY
Optimized database performance, managed backups, and implemented access-control and recovery procedures.

SKILLS
SQL, Python, PostgreSQL, AWS, Linux, Data Governance

EXPERIENCE
8 years of professional experience in database administrator responsibilities. Collaborated with cross-functional teams, documented technical decisions, and delivered measurable improvements to internal workflows.

EDUCATION
M.Sc. in Computer Applications, University of Kerala

PROJECTS
Completed practical projects involving SQL, stakeholder communication, version control, and reproducible documentation.



In [21]:

from pathlib import Path

ananya_path = Path("resumes/candidate_09.txt")
print(ananya_path.read_text(encoding="utf-8"))


Ananya Iyer
Research Scientist

SUMMARY
Published applied ML research and developed experimental pipelines for text classification and model evaluation.

SKILLS
Python, Machine Learning, Deep Learning, PyTorch, Statistics, NLP

EXPERIENCE
8 years of professional experience in research scientist responsibilities. Collaborated with cross-functional teams, documented technical decisions, and delivered measurable improvements to internal workflows.

EDUCATION
Ph.D. in Computer Science, Indian Institute of Science

PROJECTS
Completed practical projects involving Python, stakeholder communication, version control, and reproducible documentation.



In [22]:

from pathlib import Path

neha_path = Path("resumes/candidate_24.txt")
print(neha_path.read_text(encoding="utf-8"))


Neha Kulkarni
Financial Data Analyst

SUMMARY
Automated financial reports, reconciled datasets, and built dashboards for forecasting and variance analysis.

SKILLS
Python, SQL, Excel, Power BI, Statistics

EXPERIENCE
5 years of professional experience in financial data analyst responsibilities. Collaborated with cross-functional teams, documented technical decisions, and delivered measurable improvements to internal workflows.

EDUCATION
MBA in Finance, Symbiosis International University

PROJECTS
Completed practical projects involving Python, stakeholder communication, version control, and reproducible documentation.



In [23]:

from pathlib import Path

remaining_candidates = {
    "Benjamin Davis": "candidate_17.txt",
    "Sofia Garcia": "candidate_11.txt",
    "Daniel Brown": "candidate_10.txt",
}

for name, filename in remaining_candidates.items():
    path = Path("resumes") / filename
    print("\n" + "=" * 60)
    print(name, "-", filename)
    print("=" * 60)

    if path.exists():
        print(path.read_text(encoding="utf-8"))
    else:
        print("Resume not found:", path.resolve())



Benjamin Davis - candidate_17.txt
Benjamin Davis
Data Architect

SUMMARY
Designed enterprise data platforms, governance standards, and reliable lakehouse pipelines.

SKILLS
SQL, Python, Spark, AWS, Databricks, ETL, Data Governance

EXPERIENCE
10 years of professional experience in data architect responsibilities. Collaborated with cross-functional teams, documented technical decisions, and delivered measurable improvements to internal workflows.

EDUCATION
M.S. in Information Systems, Carnegie Mellon University

PROJECTS
Completed practical projects involving SQL, stakeholder communication, version control, and reproducible documentation.


Sofia Garcia - candidate_11.txt
Sofia Garcia
Business Intelligence Analyst

SUMMARY
Translated stakeholder questions into dashboards, KPI definitions, and analytical reports.

SKILLS
SQL, Power BI, Tableau, Excel, Python, Data Analysis

EXPERIENCE
5 years of professional experience in business intelligence analyst responsibilities. Collaborated wit

In [24]:

import json
from pathlib import Path

result_files = {
    "Data Scientist": "results.json",
    "RAG Engineer": "rag_results.json",
    "Data Engineer": "data_engineer_results.json",
    "BI Analyst": "bi_results.json",
    "ML Engineer": "ml_results.json",
}

for job_title, filename in result_files.items():
    path = Path(filename)

    print("\n" + "=" * 70)
    print("JOB:", job_title)
    print("=" * 70)

    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    for rank, candidate in enumerate(data.get("top_matches", [])[:10], 1):
        print(
            f"{rank}. {candidate.get('candidate_name')} | "
            f"Score: {candidate.get('match_score')} | "
            f"Experience: {candidate.get('experience_years')} | "
            f"Must-have pass: {candidate.get('must_have_pass')}"
        )
        print("   Matched:", candidate.get("matched_skills"))
        print("   Missing:", candidate.get("missing_detected_skills"))



JOB: Data Scientist
1. Aarav Sharma | Score: 70 | Experience: 5 | Must-have pass: True
   Matched: ['Machine Learning', 'Pandas', 'Python', 'SQL', 'Scikit-learn', 'Statistics']
   Missing: []
2. Emma Wilson | Score: 63 | Experience: 6 | Must-have pass: False
   Matched: ['Machine Learning', 'Pandas', 'Python', 'SQL', 'Statistics']
   Missing: ['Scikit-learn']
3. Samuel Clark | Score: 55 | Experience: 5 | Must-have pass: False
   Matched: ['Machine Learning', 'Python', 'SQL']
   Missing: ['Pandas', 'Scikit-learn', 'Statistics']
4. Fatima Khan | Score: 49 | Experience: 6 | Must-have pass: False
   Matched: ['Python', 'SQL']
   Missing: ['Machine Learning', 'Pandas', 'Scikit-learn', 'Statistics']
5. Divya Menon | Score: 49 | Experience: 8 | Must-have pass: False
   Matched: ['Python', 'SQL']
   Missing: ['Machine Learning', 'Pandas', 'Scikit-learn', 'Statistics']
6. Ananya Iyer | Score: 49 | Experience: 8 | Must-have pass: False
   Matched: ['Machine Learning', 'Python', 'Statistics']
  

In [25]:

import json
from pathlib import Path

result_files = {
    "Data Scientist": "results.json",
    "RAG Engineer": "rag_results.json",
    "Data Engineer": "data_engineer_results.json",
    "BI Analyst": "bi_results.json",
    "ML Engineer": "ml_results.json",
}

# Replace these example sets with your complete,
# human-reviewed relevance labels for each job.
ground_truth = {
    # "Data Scientist": {"candidate_01.txt", "candidate_15.txt"},
    # Add the reviewed labels for all five jobs.
}

def precision_at_k(retrieved, relevant, k=10):
    top_k = retrieved[:k]
    if not top_k:
        return 0.0
    return sum(name in relevant for name in top_k) / len(top_k)

def recall_at_k(retrieved, relevant, k=10):
    if not relevant:
        return None  # Cannot calculate recall without known positives
    return sum(name in relevant for name in retrieved[:k]) / len(relevant)

for job, filename in result_files.items():
    if job not in ground_truth:
        print(f"{job}: labels pending")
        continue

    with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)

    retrieved = [
        Path(c["resume_path"]).name
        for c in data.get("top_matches", [])
    ]

    relevant = ground_truth[job]

    p10 = precision_at_k(retrieved, relevant, 10)
    r10 = recall_at_k(retrieved, relevant, 10)

    print(f"\n{job}")
    print(f"Precision@10: {p10:.3f}")
    print(
        f"Recall@10: {r10:.3f}"
        if r10 is not None
        else "Recall@10: Not available"
    )


Data Scientist: labels pending
RAG Engineer: labels pending
Data Engineer: labels pending
BI Analyst: labels pending
ML Engineer: labels pending


In [26]:
import json
from pathlib import Path

result_files = {
    "Data Scientist": "results.json",
    "Data Engineer": "data_engineer_results.json",
    "BI Analyst": "bi_results.json",
    "ML Engineer": "ml_results.json",
    "RAG Engineer": "rag_results.json",
}

for job_name, filename in result_files.items():
    path = Path(filename)

    if not path.exists():
        print(f"\nMissing file: {filename}")
        continue

    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    print(f"\n{'=' * 65}")
    print(job_name)
    print(f"{'=' * 65}")

    for i, candidate in enumerate(data["top_matches"], start=1):
        print(
            f"{i:2}. {candidate['candidate_name']} | "
            f"Score: {candidate['match_score']} | "
            f"Experience: {candidate.get('experience_years')} | "
            f"Must-have pass: {candidate.get('must_have_pass')}"
        )
        print("    Matched:", candidate.get("matched_skills", []))
        print("    Missing:", candidate.get("missing_detected_skills", []))


Data Scientist
 1. Aarav Sharma | Score: 70 | Experience: 5 | Must-have pass: True
    Matched: ['Machine Learning', 'Pandas', 'Python', 'SQL', 'Scikit-learn', 'Statistics']
    Missing: []
 2. Emma Wilson | Score: 63 | Experience: 6 | Must-have pass: False
    Matched: ['Machine Learning', 'Pandas', 'Python', 'SQL', 'Statistics']
    Missing: ['Scikit-learn']
 3. Samuel Clark | Score: 55 | Experience: 5 | Must-have pass: False
    Matched: ['Machine Learning', 'Python', 'SQL']
    Missing: ['Pandas', 'Scikit-learn', 'Statistics']
 4. Fatima Khan | Score: 49 | Experience: 6 | Must-have pass: False
    Matched: ['Python', 'SQL']
    Missing: ['Machine Learning', 'Pandas', 'Scikit-learn', 'Statistics']
 5. Divya Menon | Score: 49 | Experience: 8 | Must-have pass: False
    Matched: ['Python', 'SQL']
    Missing: ['Machine Learning', 'Pandas', 'Scikit-learn', 'Statistics']
 6. Ananya Iyer | Score: 49 | Experience: 8 | Must-have pass: False
    Matched: ['Machine Learning', 'Python', 'Sta

In [27]:
ground_truth = {
    "Data Scientist": {
        "candidate_01.txt": 1,
        "candidate_15.txt": 1,
        # Add reviewed candidates here.
    },
    "Data Engineer": {
        # Add reviewed candidates here.
    },
    "BI Analyst": {
        # Add reviewed candidates here.
    },
    "ML Engineer": {
        # Add reviewed candidates here.
    },
    "RAG Engineer": {
        # Add reviewed candidates here.
    },
}

In [28]:
def precision_at_k(ranked_candidates, labels, k=10):
    top_k = ranked_candidates[:k]

    if not top_k:
        return 0.0

    relevant_count = sum(
        labels.get(candidate["resume_path"].replace("\\", "/").split("/")[-1], 0)
        for candidate in top_k
    )

    return relevant_count / len(top_k)


for job_name, filename in result_files.items():
    with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)

    labels = ground_truth.get(job_name, {})

    score = precision_at_k(data["top_matches"], labels, k=10)
    print(f"{job_name}: Precision@10 = {score:.2f}")

Data Scientist: Precision@10 = 0.20
Data Engineer: Precision@10 = 0.00
BI Analyst: Precision@10 = 0.00
ML Engineer: Precision@10 = 0.00
RAG Engineer: Precision@10 = 0.00


In [29]:
import json
from pathlib import Path

result_files = {
    "Data Scientist": "results.json",
    "Data Engineer": "data_engineer_results.json",
    "BI Analyst": "bi_results.json",
    "ML Engineer": "ml_results.json",
    "RAG Engineer": "rag_results.json",
}

for job, filename in result_files.items():
    with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(f"\n{'=' * 50}\n{job}\n{'=' * 50}")

    for i, c in enumerate(data["top_matches"], start=1):
        print(
            f"{i}. {c['candidate_name']} | "
            f"{c['resume_path']} | Score: {c['match_score']}"
        )


Data Scientist
1. Aarav Sharma | resumes/candidate_01.txt | Score: 70
2. Emma Wilson | resumes/candidate_15.txt | Score: 63
3. Samuel Clark | resumes/candidate_25.txt | Score: 55
4. Fatima Khan | resumes/candidate_07.txt | Score: 49
5. Divya Menon | resumes/candidate_28.txt | Score: 49
6. Ananya Iyer | resumes/candidate_09.txt | Score: 49
7. Neha Kulkarni | resumes/candidate_24.txt | Score: 49
8. Benjamin Davis | resumes/candidate_17.txt | Score: 48
9. Sofia Garcia | resumes/candidate_11.txt | Score: 45
10. Daniel Brown | resumes/candidate_10.txt | Score: 42

Data Engineer
1. Fatima Khan | resumes/candidate_07.txt | Score: 72
2. Samuel Clark | resumes/candidate_25.txt | Score: 66
3. Benjamin Davis | resumes/candidate_17.txt | Score: 66
4. Ethan Johnson | resumes/candidate_06.txt | Score: 49
5. Divya Menon | resumes/candidate_28.txt | Score: 49
6. Jack Thompson | resumes/candidate_29.txt | Score: 48
7. Neha Kulkarni | resumes/candidate_24.txt | Score: 47
8. Emma Wilson | resumes/candid

In [30]:

ground_truth = {
    "Data Scientist": {
        "candidate_01.txt": 1,  # Aarav Sharma
        "candidate_15.txt": 1,  # Emma Wilson
        "candidate_25.txt": 0,  # Samuel Clark
        "candidate_07.txt": 0,  # Fatima Khan
        "candidate_28.txt": 0,  # Divya Menon
        "candidate_09.txt": 1,  # Ananya Iyer
        "candidate_24.txt": 0,  # Neha Kulkarni
        # Review the remaining three candidates before adding labels.
    },

    "Data Engineer": {
        # Add reviewed candidate labels here.
    },

    "BI Analyst": {
        # Add reviewed candidate labels here.
    },

    "ML Engineer": {
        # Add reviewed candidate labels here.
    },

    "RAG Engineer": {
        # Add reviewed candidate labels here.
    },
}


In [31]:

import json
from pathlib import Path

result_files = {
    "Data Scientist": "results.json",
    "Data Engineer": "data_engineer_results.json",
    "BI Analyst": "bi_results.json",
    "ML Engineer": "ml_results.json",
    "RAG Engineer": "rag_results.json",
}

for job, filename in result_files.items():
    path = Path(filename)

    print(f"\n{'=' * 60}")
    print(f"JOB: {job}")
    print(f"{'=' * 60}")

    if not path.exists():
        print(f"File not found: {path.resolve()}")
        continue

    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)

    for i, candidate in enumerate(data.get("top_matches", []), start=1):
        print(
            f"\n{i}. {candidate.get('candidate_name')} "
            f"| File: {Path(candidate.get('resume_path', '')).name} "
            f"| Score: {candidate.get('match_score')}"
        )
        print("   Matched skills:", candidate.get("matched_skills", []))
        print("   Missing skills:", candidate.get("missing_detected_skills", []))
        print("   Must-have pass:", candidate.get("must_have_pass"))



JOB: Data Scientist

1. Aarav Sharma | File: candidate_01.txt | Score: 70
   Matched skills: ['Machine Learning', 'Pandas', 'Python', 'SQL', 'Scikit-learn', 'Statistics']
   Missing skills: []
   Must-have pass: True

2. Emma Wilson | File: candidate_15.txt | Score: 63
   Matched skills: ['Machine Learning', 'Pandas', 'Python', 'SQL', 'Statistics']
   Missing skills: ['Scikit-learn']
   Must-have pass: False

3. Samuel Clark | File: candidate_25.txt | Score: 55
   Matched skills: ['Machine Learning', 'Python', 'SQL']
   Missing skills: ['Pandas', 'Scikit-learn', 'Statistics']
   Must-have pass: False

4. Fatima Khan | File: candidate_07.txt | Score: 49
   Matched skills: ['Python', 'SQL']
   Missing skills: ['Machine Learning', 'Pandas', 'Scikit-learn', 'Statistics']
   Must-have pass: False

5. Divya Menon | File: candidate_28.txt | Score: 49
   Matched skills: ['Python', 'SQL']
   Missing skills: ['Machine Learning', 'Pandas', 'Scikit-learn', 'Statistics']
   Must-have pass: False



In [32]:
# Example: print only the Data Engineer candidates
job = "Data Engineer"
filename = result_files[job]

In [33]:

import json
from pathlib import Path

job = "Data Engineer"
filename = result_files[job]

with open(filename, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"\n{'=' * 50}")
print(f"JOB: {job}")
print(f"{'=' * 50}")

for i, candidate in enumerate(data["top_matches"], start=1):
    print(
        f"\n{i}. {candidate['candidate_name']} "
        f"| File: {Path(candidate['resume_path']).name} "
        f"| Score: {candidate['match_score']}"
    )
    print("   Matched skills:", candidate.get("matched_skills", []))
    print("   Missing skills:", candidate.get("missing_detected_skills", []))
    print("   Must-have pass:", candidate.get("must_have_pass"))



JOB: Data Engineer

1. Fatima Khan | File: candidate_07.txt | Score: 72
   Matched skills: ['AWS', 'Airflow', 'Databricks', 'ETL', 'Python', 'SQL', 'Spark']
   Missing skills: []
   Must-have pass: True

2. Samuel Clark | File: candidate_25.txt | Score: 66
   Matched skills: ['AWS', 'Airflow', 'ETL', 'Python', 'SQL', 'Spark']
   Missing skills: ['Databricks']
   Must-have pass: False

3. Benjamin Davis | File: candidate_17.txt | Score: 66
   Matched skills: ['AWS', 'Databricks', 'ETL', 'Python', 'SQL', 'Spark']
   Missing skills: ['Airflow']
   Must-have pass: False

4. Ethan Johnson | File: candidate_06.txt | Score: 49
   Matched skills: ['AWS', 'Python']
   Missing skills: ['Airflow', 'Databricks', 'ETL', 'SQL', 'Spark']
   Must-have pass: False

5. Divya Menon | File: candidate_28.txt | Score: 49
   Matched skills: ['AWS', 'Python', 'SQL']
   Missing skills: ['Airflow', 'Databricks', 'ETL', 'Spark']
   Must-have pass: False

6. Jack Thompson | File: candidate_29.txt | Score: 48
   

In [34]:

ground_truth = {
    "Data Scientist": {
        "candidate_01.txt": 1,  # Aarav Sharma
        "candidate_15.txt": 1,  # Emma Wilson
        "candidate_25.txt": 0,  # Samuel Clark: provisional
        "candidate_07.txt": 0,  # Fatima Khan: provisional for DS
        "candidate_28.txt": 0,  # Divya Menon
        "candidate_09.txt": 1,  # Ananya Iyer
        "candidate_24.txt": 0,  # Neha Kulkarni
    },

    "Data Engineer": {
        "candidate_07.txt": 1,  # Fatima Khan: strong skills match
        "candidate_25.txt": 1,  # Samuel Clark: relevant ETL/data engineering
        "candidate_17.txt": 1,  # Benjamin Davis: relevant data architecture
        "candidate_06.txt": 0,  # Ethan Johnson: several core skills missing
        # Review candidates 5-10 before adding more labels.
    },

    "BI Analyst": {
        # Add reviewed labels after printing candidates.
    },

    "ML Engineer": {
        # Add reviewed labels after printing candidates.
    },

    "RAG Engineer": {
        # Add reviewed labels after printing candidates.
    },
}

print("Ground-truth labels updated.")


Ground-truth labels updated.


In [35]:

job = "BI Analyst"
filename = result_files[job]

with open(filename, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"\n{'=' * 50}\nJOB: {job}\n{'=' * 50}")

for i, c in enumerate(data["top_matches"], start=1):
    print(
        f"\n{i}. {c['candidate_name']} "
        f"| File: {Path(c['resume_path']).name} "
        f"| Score: {c['match_score']}"
    )
    print("   Matched:", c.get("matched_skills", []))
    print("   Missing:", c.get("missing_detected_skills", []))



JOB: BI Analyst

1. Priya Reddy | File: candidate_04.txt | Score: 69
   Matched: ['Excel', 'Power BI', 'Python', 'SQL', 'Statistics', 'Tableau']
   Missing: ['Data Analysis']

2. Sofia Garcia | File: candidate_11.txt | Score: 68
   Matched: ['Data Analysis', 'Excel', 'Power BI', 'Python', 'SQL', 'Tableau']
   Missing: ['Statistics']

3. Sana Qureshi | File: candidate_30.txt | Score: 63
   Matched: ['Data Analysis', 'Excel', 'Power BI', 'Python', 'SQL', 'Statistics']
   Missing: ['Tableau']

4. Neha Kulkarni | File: candidate_24.txt | Score: 62
   Matched: ['Excel', 'Power BI', 'Python', 'SQL', 'Statistics']
   Missing: ['Data Analysis', 'Tableau']

5. Emma Wilson | File: candidate_15.txt | Score: 52
   Matched: ['Python', 'SQL', 'Statistics']
   Missing: ['Data Analysis', 'Excel', 'Power BI', 'Tableau']

6. Leo Fischer | File: candidate_27.txt | Score: 50
   Matched: ['Python', 'SQL', 'Statistics']
   Missing: ['Data Analysis', 'Excel', 'Power BI', 'Tableau']

7. Mei Lin | File: candi

In [36]:

ground_truth["BI Analyst"] = {
    "candidate_04.txt": 1,  # Priya Reddy: strong skills match
    "candidate_11.txt": 1,  # Sofia Garcia: strong skills match
    "candidate_30.txt": 1,  # Sana Qureshi: strong skills match
    "candidate_24.txt": 1,  # Neha Kulkarni: relevant analysis skills
    "candidate_15.txt": 0,  # Emma Wilson: several BI tools missing
}

print("BI Analyst labels added.")


BI Analyst labels added.


In [37]:

job = "ML Engineer"
filename = result_files[job]

with open(filename, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"\n{'=' * 50}\nJOB: {job}\n{'=' * 50}")

for i, c in enumerate(data["top_matches"], start=1):
    print(
        f"\n{i}. {c['candidate_name']} "
        f"| File: {Path(c['resume_path']).name} "
        f"| Score: {c['match_score']}"
    )
    print("   Matched:", c.get("matched_skills", []))
    print("   Missing:", c.get("missing_detected_skills", []))



JOB: ML Engineer

1. Maya Patel | File: candidate_02.txt | Score: 61
   Matched: ['AWS', 'Docker', 'PyTorch', 'Python', 'TensorFlow']
   Missing: ['Azure', 'Machine Learning']

2. Ethan Johnson | File: candidate_06.txt | Score: 54
   Matched: ['AWS', 'Azure', 'Docker', 'Python']
   Missing: ['Machine Learning', 'PyTorch', 'TensorFlow']

3. Samuel Clark | File: candidate_25.txt | Score: 52
   Matched: ['AWS', 'Machine Learning', 'Python']
   Missing: ['Azure', 'Docker', 'PyTorch', 'TensorFlow']

4. Ananya Iyer | File: candidate_09.txt | Score: 51
   Matched: ['Machine Learning', 'PyTorch', 'Python']
   Missing: ['AWS', 'Azure', 'Docker', 'TensorFlow']

5. Rohan Desai | File: candidate_12.txt | Score: 49
   Matched: ['AWS', 'Docker', 'Python']
   Missing: ['Azure', 'Machine Learning', 'PyTorch', 'TensorFlow']

6. Leo Fischer | File: candidate_27.txt | Score: 49
   Matched: ['Machine Learning', 'PyTorch', 'Python']
   Missing: ['AWS', 'Azure', 'Docker', 'TensorFlow']

7. Noah Williams | 

In [38]:

ground_truth["ML Engineer"] = {
    "candidate_02.txt": 1,  # Maya Patel: PyTorch, TensorFlow, Docker, AWS
    "candidate_06.txt": 0,  # Ethan Johnson: ML frameworks not detected
    "candidate_25.txt": 1,  # Samuel Clark: ML experience and Python
    "candidate_09.txt": 1,  # Ananya Iyer: ML and PyTorch
    "candidate_12.txt": 0,  # Rohan Desai: ML frameworks not detected
    "candidate_07.txt": 0,  # Fatima Khan: primarily data engineering
}

print("ML Engineer labels added.")


ML Engineer labels added.


In [40]:

job = "RAG Engineer"
filename = result_files[job]

with open(filename, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"\n{'=' * 50}\nJOB: {job}\n{'=' * 50}")

for i, c in enumerate(data["top_matches"], start=1):
    print(
        f"\n{i}. {c['candidate_name']} "
        f"| File: {Path(c['resume_path']).name} "
        f"| Score: {c['match_score']}"
    )
    print("   Matched:", c.get("matched_skills", []))
    print("   Missing:", c.get("missing_detected_skills", []))



JOB: RAG Engineer

1. Olivia Chen | File: candidate_05.txt | Score: 67
   Matched: ['ChromaDB', 'FastAPI', 'LangChain', 'Python', 'RAG']
   Missing: []

2. Arjun Rao | File: candidate_21.txt | Score: 66
   Matched: ['ChromaDB', 'FastAPI', 'LangChain', 'Python', 'RAG']
   Missing: []

3. Rohan Desai | File: candidate_12.txt | Score: 42
   Matched: ['FastAPI', 'Python']
   Missing: ['ChromaDB', 'LangChain', 'RAG']

4. Noah Williams | File: candidate_03.txt | Score: 41
   Matched: ['FastAPI', 'Python']
   Missing: ['ChromaDB', 'LangChain', 'RAG']

5. Mei Lin | File: candidate_20.txt | Score: 40
   Matched: ['Python']
   Missing: ['ChromaDB', 'FastAPI', 'LangChain', 'RAG']

6. Ananya Iyer | File: candidate_09.txt | Score: 38
   Matched: ['Python']
   Missing: ['ChromaDB', 'FastAPI', 'LangChain', 'RAG']

7. Leo Fischer | File: candidate_27.txt | Score: 37
   Matched: ['Python']
   Missing: ['ChromaDB', 'FastAPI', 'LangChain', 'RAG']

8. Samuel Clark | File: candidate_25.txt | Score: 37
   

In [41]:

ground_truth["RAG Engineer"] = {
    "candidate_05.txt": 1,  # Olivia Chen: all detected skills matched
    "candidate_21.txt": 1,  # Arjun Rao: all detected skills matched
    "candidate_12.txt": 0,  # Rohan Desai: core RAG skills missing
    "candidate_03.txt": 0,  # Noah Williams: core RAG skills missing
    "candidate_20.txt": 0,  # Mei Lin: core RAG skills missing
    "candidate_16.txt": 0,  # Kavya Nair: core RAG skills missing
}

print("RAG Engineer labels added.")


RAG Engineer labels added.


In [42]:

import json
from pathlib import Path

def precision_at_k(ranked_candidates, labels, k=10):
    top_k = ranked_candidates[:k]

    if not top_k:
        return 0.0

    relevant = 0

    for candidate in top_k:
        filename = Path(candidate["resume_path"]).name
        relevant += labels.get(filename, 0)

    return relevant / len(top_k)


print("=" * 55)
print("PRECISION@10 RESULTS")
print("=" * 55)

for job, filename in result_files.items():
    with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)

    labels = ground_truth.get(job, {})
    precision = precision_at_k(data["top_matches"], labels, k=10)

    print(
        f"{job}: Precision@10 = {precision:.2f} "
        f"({precision * 100:.0f}%)"
    )


PRECISION@10 RESULTS
Data Scientist: Precision@10 = 0.30 (30%)
Data Engineer: Precision@10 = 0.30 (30%)
BI Analyst: Precision@10 = 0.40 (40%)
ML Engineer: Precision@10 = 0.30 (30%)
RAG Engineer: Precision@10 = 0.20 (20%)


In [43]:

import json
from pathlib import Path

print("GROUND-TRUTH LABEL COVERAGE")
print("=" * 55)

for job, filename in result_files.items():
    with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)

    candidates = data["top_matches"][:10]
    labels = ground_truth.get(job, {})

    labeled = sum(
        Path(c["resume_path"]).name in labels
        for c in candidates
    )

    print(
        f"{job}: {labeled}/{len(candidates)} candidates labeled"
    )


GROUND-TRUTH LABEL COVERAGE
Data Scientist: 7/10 candidates labeled
Data Engineer: 4/10 candidates labeled
BI Analyst: 5/10 candidates labeled
ML Engineer: 6/10 candidates labeled
RAG Engineer: 6/10 candidates labeled


In [44]:

import json
from pathlib import Path

print("=" * 60)
print("CANDIDATES STILL NEEDING GROUND-TRUTH LABELS")
print("=" * 60)

for job, filename in result_files.items():
    with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)

    labels = ground_truth.get(job, {})
    candidates = data["top_matches"][:10]

    unlabeled = [
        c for c in candidates
        if Path(c["resume_path"]).name not in labels
    ]

    print(f"\n{job}: {len(unlabeled)} unlabeled")

    for c in unlabeled:
        print(
            f"  {c['candidate_name']} | "
            f"{Path(c['resume_path']).name} | "
            f"Score: {c['match_score']}"
        )
        print("    Matched:", c.get("matched_skills", []))
        print("    Missing:", c.get("missing_detected_skills", []))


CANDIDATES STILL NEEDING GROUND-TRUTH LABELS

Data Scientist: 3 unlabeled
  Benjamin Davis | candidate_17.txt | Score: 48
    Matched: ['Python', 'SQL']
    Missing: ['Machine Learning', 'Pandas', 'Scikit-learn', 'Statistics']
  Sofia Garcia | candidate_11.txt | Score: 45
    Matched: ['Python', 'SQL']
    Missing: ['Machine Learning', 'Pandas', 'Scikit-learn', 'Statistics']
  Daniel Brown | candidate_10.txt | Score: 42
    Matched: ['SQL']
    Missing: ['Machine Learning', 'Pandas', 'Python', 'Scikit-learn', 'Statistics']

Data Engineer: 6 unlabeled
  Divya Menon | candidate_28.txt | Score: 49
    Matched: ['AWS', 'Python', 'SQL']
    Missing: ['Airflow', 'Databricks', 'ETL', 'Spark']
  Jack Thompson | candidate_29.txt | Score: 48
    Matched: ['AWS', 'Python']
    Missing: ['Airflow', 'Databricks', 'ETL', 'SQL', 'Spark']
  Neha Kulkarni | candidate_24.txt | Score: 47
    Matched: ['Python', 'SQL']
    Missing: ['AWS', 'Airflow', 'Databricks', 'ETL', 'Spark']
  Emma Wilson | candidate

In [45]:

# Provisional labels for candidates that still need review

ground_truth["Data Scientist"].update({
    "candidate_17.txt": 0,  # Benjamin Davis: data architecture
    "candidate_11.txt": 0,  # Sofia Garcia: business analysis
    "candidate_10.txt": 0,  # Daniel Brown: core DS skills not detected
})

ground_truth["Data Engineer"].update({
    "candidate_28.txt": 0,  # Divya Menon: database administration
    "candidate_29.txt": 0,  # Jack Thompson: core DE skills not detected
    "candidate_24.txt": 0,  # Neha Kulkarni: financial analysis
})

print("Provisional labels added.")


Provisional labels added.


In [46]:

import json
from pathlib import Path

for job, filename in result_files.items():
    with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)

    labels = ground_truth.get(job, {})

    unlabeled = [
        c for c in data["top_matches"][:10]
        if Path(c["resume_path"]).name not in labels
    ]

    print(f"\n--- {job} ---")

    for c in unlabeled:
        print(
            f"{c['candidate_name']} | "
            f"{Path(c['resume_path']).name} | "
            f"Matched: {c.get('matched_skills', [])} | "
            f"Missing: {c.get('missing_detected_skills', [])}"
        )



--- Data Scientist ---

--- Data Engineer ---
Emma Wilson | candidate_15.txt | Matched: ['Python', 'SQL'] | Missing: ['AWS', 'Airflow', 'Databricks', 'ETL', 'Spark']
Aarav Sharma | candidate_01.txt | Matched: ['Python', 'SQL'] | Missing: ['AWS', 'Airflow', 'Databricks', 'ETL', 'Spark']
Rohan Desai | candidate_12.txt | Matched: ['AWS', 'Python'] | Missing: ['Airflow', 'Databricks', 'ETL', 'SQL', 'Spark']

--- BI Analyst ---
Leo Fischer | candidate_27.txt | Matched: ['Python', 'SQL', 'Statistics'] | Missing: ['Data Analysis', 'Excel', 'Power BI', 'Tableau']
Mei Lin | candidate_20.txt | Matched: ['Data Analysis', 'Python', 'SQL'] | Missing: ['Excel', 'Power BI', 'Statistics', 'Tableau']
Aarav Sharma | candidate_01.txt | Matched: ['Python', 'SQL', 'Statistics'] | Missing: ['Data Analysis', 'Excel', 'Power BI', 'Tableau']
Benjamin Davis | candidate_17.txt | Matched: ['Python', 'SQL'] | Missing: ['Data Analysis', 'Excel', 'Power BI', 'Statistics', 'Tableau']
Grace Kim | candidate_13.txt | M

In [47]:

# Data Engineer
ground_truth["Data Engineer"].update({
    "candidate_15.txt": 0,  # Emma Wilson: BI/product data science profile
    "candidate_01.txt": 0,  # Aarav Sharma: data scientist profile
    "candidate_12.txt": 0,  # Rohan Desai: core DE skills not detected
})

# BI Analyst
ground_truth["BI Analyst"].update({
    "candidate_27.txt": 0,  # Leo Fischer: core BI tools not detected
    "candidate_20.txt": 0,  # Mei Lin: Excel and BI tools not detected
    "candidate_01.txt": 0,  # Aarav Sharma: data science profile
    "candidate_17.txt": 0,  # Benjamin Davis: data architecture profile
    "candidate_13.txt": 0,  # Grace Kim: core BI tools not detected
})

# ML Engineer
ground_truth["ML Engineer"].update({
    "candidate_27.txt": 1,  # Leo Fischer: ML and PyTorch
    "candidate_03.txt": 0,  # Noah Williams: ML frameworks not detected
    "candidate_29.txt": 0,  # Jack Thompson: ML frameworks not detected
    "candidate_23.txt": 0,  # Michael Osei: ML frameworks not detected
})

# RAG Engineer
ground_truth["RAG Engineer"].update({
    "candidate_09.txt": 0,  # Ananya Iyer: RAG stack not detected
    "candidate_27.txt": 0,  # Leo Fischer: RAG stack not detected
    "candidate_25.txt": 0,  # Samuel Clark: RAG stack not detected
    "candidate_11.txt": 0,  # Sofia Garcia: RAG stack not detected
})

print("Provisional labels added.")


Provisional labels added.


In [48]:

import json
from pathlib import Path

print("GROUND-TRUTH LABEL COVERAGE")
print("=" * 55)

for job, filename in result_files.items():
    with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)

    labels = ground_truth.get(job, {})
    candidates = data["top_matches"][:10]

    labeled = sum(
        Path(c["resume_path"]).name in labels
        for c in candidates
    )

    print(f"{job}: {labeled}/{len(candidates)} candidates labeled")


GROUND-TRUTH LABEL COVERAGE
Data Scientist: 10/10 candidates labeled
Data Engineer: 10/10 candidates labeled
BI Analyst: 10/10 candidates labeled
ML Engineer: 10/10 candidates labeled
RAG Engineer: 10/10 candidates labeled


In [49]:

import json
from pathlib import Path

def precision_at_k(ranked_candidates, labels, k=10):
    top_k = ranked_candidates[:k]

    if not top_k:
        return 0.0

    relevant = sum(
        labels.get(Path(c["resume_path"]).name, 0)
        for c in top_k
    )

    return relevant / len(top_k)


print("=" * 55)
print("FINAL PRECISION@10 RESULTS")
print("=" * 55)

for job, filename in result_files.items():
    with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)

    precision = precision_at_k(
        data["top_matches"],
        ground_truth.get(job, {}),
        k=10
    )

    print(f"{job}: {precision:.2f} ({precision * 100:.0f}%)")


FINAL PRECISION@10 RESULTS
Data Scientist: 0.30 (30%)
Data Engineer: 0.30 (30%)
BI Analyst: 0.40 (40%)
ML Engineer: 0.40 (40%)
RAG Engineer: 0.20 (20%)


In [50]:

import csv
import json
from pathlib import Path

evaluation_rows = []

for job, filename in result_files.items():
    with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)

    candidates = data["top_matches"][:10]
    labels = ground_truth.get(job, {})

    relevant = sum(
        labels.get(Path(c["resume_path"]).name, 0)
        for c in candidates
    )

    evaluation_rows.append({
        "job_role": job,
        "k": len(candidates),
        "relevant_in_top_k": relevant,
        "precision_at_10": relevant / len(candidates) if candidates else 0,
        "precision_percent": (
            100 * relevant / len(candidates) if candidates else 0
        ),
        "label_coverage": sum(
            Path(c["resume_path"]).name in labels for c in candidates
        ),
    })

output_file = Path("evaluation_results.csv")

with output_file.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=evaluation_rows[0].keys()
    )
    writer.writeheader()
    writer.writerows(evaluation_rows)

print("Saved:", output_file.resolve())
for row in evaluation_rows:
    print(row)


Saved: C:\Users\HP\Desktop\AIRTRIBE\RAG project\resume_rag_job_matcher\evaluation_results.csv
{'job_role': 'Data Scientist', 'k': 10, 'relevant_in_top_k': 3, 'precision_at_10': 0.3, 'precision_percent': 30.0, 'label_coverage': 10}
{'job_role': 'Data Engineer', 'k': 10, 'relevant_in_top_k': 3, 'precision_at_10': 0.3, 'precision_percent': 30.0, 'label_coverage': 10}
{'job_role': 'BI Analyst', 'k': 10, 'relevant_in_top_k': 4, 'precision_at_10': 0.4, 'precision_percent': 40.0, 'label_coverage': 10}
{'job_role': 'ML Engineer', 'k': 10, 'relevant_in_top_k': 4, 'precision_at_10': 0.4, 'precision_percent': 40.0, 'label_coverage': 10}
{'job_role': 'RAG Engineer', 'k': 10, 'relevant_in_top_k': 2, 'precision_at_10': 0.2, 'precision_percent': 20.0, 'label_coverage': 10}


In [51]:

from pathlib import Path
import json
import pandas as pd

# Paths to your existing result files
result_files = {
    "Data Scientist": "results.json",
    "Data Engineer": "data_engineer_results.json",
    "BI Analyst": "bi_results.json",
    "ML Engineer": "ml_results.json",
    "RAG Engineer": "rag_results.json",
}

# Display the first candidate's available information
for role, filename in result_files.items():
    path = Path(filename)

    if not path.exists():
        print(f"File not found: {filename}")
        continue

    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    print(f"\n{'=' * 50}")
    print("ROLE:", role)
    print("FILE:", filename)
    print("DATA TYPE:", type(data).__name__)

    if isinstance(data, list) and data:
        print("FIRST CANDIDATE:")
        print(json.dumps(data[0], indent=2)[:2500])
    elif isinstance(data, dict):
        print("AVAILABLE KEYS:", list(data.keys()))
        print(json.dumps(data, indent=2)[:2500])



ROLE: Data Scientist
FILE: results.json
DATA TYPE: dict
AVAILABLE KEYS: ['job_description', 'filters', 'top_matches', 'metrics', 'scoring_note']
{
  "job_description": "Data Scientist\nWe are looking for a Data Scientist with 5+ years of experience. Must have Python, SQL, machine learning, statistics, Pandas, and Scikit-learn. Experience building predictive models, running experiments, and communicating findings to product stakeholders is important. Cloud deployment experience is a plus.\n",
  "filters": {
    "minimum_experience_years_detected": 5,
    "must_have_skills_detected": [
      "Python",
      "SQL",
      "Machine Learning",
      "Scikit-learn",
      "Pandas",
      "Statistics"
    ]
  },
  "top_matches": [
    {
      "candidate_name": "Aarav Sharma",
      "resume_path": "resumes/candidate_01.txt",
      "match_score": 70,
      "matched_skills": [
        "Machine Learning",
        "Pandas",
        "Python",
        "SQL",
        "Scikit-learn",
        "Statisti

In [52]:

import json
from pathlib import Path
import pandas as pd

result_files = {
    "Data Scientist": "results.json",
    "Data Engineer": "data_engineer_results.json",
    "BI Analyst": "bi_results.json",
    "ML Engineer": "ml_results.json",
    "RAG Engineer": "rag_results.json",
}

rows = []

for role, filename in result_files.items():
    with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)

    for candidate in data["top_matches"]:
        resume_file = Path(candidate["resume_path"]).name

        rows.append({
            "Job Role": role,
            "Candidate": candidate["candidate_name"],
            "Resume File": resume_file,
            "Score": candidate["match_score"],
            "Experience": candidate.get("experience_years"),
            "Matched Skills": ", ".join(candidate.get("matched_skills", [])),
            "Missing Skills": ", ".join(
                candidate.get("missing_detected_skills", [])
            ),
            "Must-Have Pass": candidate.get("must_have_pass"),
            "Current Label": ground_truth.get(role, {}).get(
                resume_file, "UNLABELED"
            ),
            "Evidence": " | ".join(
                candidate.get("relevant_excerpts", [])
            ),
        })

review_df = pd.DataFrame(rows)

# Show all candidates with the current label of 0 first
display(
    review_df[review_df["Current Label"] == 0][
        ["Job Role", "Candidate", "Score", "Experience",
         "Matched Skills", "Missing Skills", "Must-Have Pass"]
    ].sort_values(["Job Role", "Score"], ascending=[True, False])
)


,Job Role,Candidate,Score,Experience,Matched Skills,Missing Skills,Must-Have Pass
24,BI Analyst,Emma Wilson,52,6,"Python, SQL, Statistics","Data Analysis, Excel, Power BI, Tableau",False
25,BI Analyst,Leo Fischer,50,4,"Python, SQL, Statistics","Data Analysis, Excel, Power BI, Tableau",False
26,BI Analyst,Mei Lin,48,4,"Data Analysis, Python, SQL","Excel, Power BI, Statistics, Tableau",False
27,BI Analyst,Aarav Sharma,47,5,"Python, SQL, Statistics","Data Analysis, Excel, Power BI, Tableau",False
28,BI Analyst,Benjamin Davis,44,10,"Python, SQL","Data Analysis, Excel, Power BI, Statistics, Ta...",False
29,BI Analyst,Grace Kim,44,4,"Python, SQL","Data Analysis, Excel, Power BI, Statistics, Ta...",False
13,Data Engineer,Ethan Johnson,49,7,"AWS, Python","Airflow, Databricks, ETL, SQL, Spark",False
14,Data Engineer,Divya Menon,49,8,"AWS, Python, SQL","Airflow, Databricks, ETL, Spark",False
15,Data Engineer,Jack Thompson,48,5,"AWS, Python","Airflow, Databricks, ETL, SQL, Spark",False
16,Data Engineer,Neha Kulkarni,47,5,"Python, SQL","AWS, Airflow, Databricks, ETL, Spark",False


In [53]:

def review_candidate(role, resume_file):
    filename = result_files[role]

    with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)

    candidate = next(
        (
            c for c in data["top_matches"]
            if Path(c["resume_path"]).name == resume_file
        ),
        None
    )

    if candidate is None:
        print("Candidate not found in this role's top 10.")
        return

    resume_path = Path(candidate["resume_path"])

    if not resume_path.exists():
        resume_path = Path("resumes") / resume_file

    print("JOB ROLE:", role)
    print("CANDIDATE:", candidate["candidate_name"])
    print("SCORE:", candidate["match_score"])
    print("CURRENT LABEL:", ground_truth.get(role, {}).get(
        resume_file, "UNLABELED"
    ))
    print("\nJOB DESCRIPTION:\n")
    print(data["job_description"])

    print("\nMATCHED SKILLS:")
    print(candidate.get("matched_skills", []))

    print("\nMISSING DETECTED SKILLS:")
    print(candidate.get("missing_detected_skills", []))

    print("\nFULL RESUME:\n")
    if resume_path.exists():
        print(resume_path.read_text(encoding="utf-8", errors="replace"))
    else:
        print("Resume file not found at:", resume_path)

# Example: inspect Priya Reddy for the BI Analyst role
review_candidate("BI Analyst", "candidate_04.txt")


JOB ROLE: BI Analyst
CANDIDATE: Priya Reddy
SCORE: 69
CURRENT LABEL: 1

JOB DESCRIPTION:

Business Intelligence Analyst
Seeking a BI Analyst with 3+ years of experience using SQL, Power BI or Tableau, Excel, and data analysis. Must be able to build dashboards, define KPIs, explain trends to stakeholders, and validate source data. Python and statistics are desirable.


MATCHED SKILLS:
['Excel', 'Power BI', 'Python', 'SQL', 'Statistics', 'Tableau']

MISSING DETECTED SKILLS:
['Data Analysis']

FULL RESUME:

Priya Reddy
Data Analyst

SUMMARY
Created executive dashboards, analyzed sales funnels, and automated recurring reports with Python and SQL.

SKILLS
SQL, Python, Excel, Tableau, Power BI, Statistics

EXPERIENCE
3 years of professional experience in data analyst responsibilities. Collaborated with cross-functional teams, documented technical decisions, and delivered measurable improvements to internal workflows.

EDUCATION
B.Sc. in Statistics, Andhra University

PROJECTS
Completed pract

In [54]:

# Create a manual-review copy without changing ground_truth

review_df["Verified Label"] = None
review_df["Review Notes"] = ""

# Start with candidates currently labeled 0
needs_review = review_df[
    review_df["Current Label"] == 0
].copy()

needs_review = needs_review[
    [
        "Job Role",
        "Candidate",
        "Resume File",
        "Score",
        "Experience",
        "Matched Skills",
        "Missing Skills",
        "Evidence",
        "Current Label",
        "Verified Label",
        "Review Notes",
    ]
]

display(needs_review.reset_index(drop=True))


,Job Role,Candidate,Resume File,Score,Experience,Matched Skills,Missing Skills,Evidence,Current Label,Verified Label,Review Notes
0,Data Scientist,Samuel Clark,candidate_25.txt,55,5,"Machine Learning, Python, SQL","Pandas, Scikit-learn, Statistics","Python, SQL, Spark, Machine Learning, AWS, Air...",0,None,
1,Data Scientist,Fatima Khan,candidate_07.txt,49,6,"Python, SQL","Machine Learning, Pandas, Scikit-learn, Statis...",6 years of professional experience in data eng...,0,None,
2,Data Scientist,Divya Menon,candidate_28.txt,49,8,"Python, SQL","Machine Learning, Pandas, Scikit-learn, Statis...","SQL, Python, PostgreSQL, AWS, Linux, Data Gove...",0,None,
3,Data Scientist,Neha Kulkarni,candidate_24.txt,49,5,"Python, SQL, Statistics","Machine Learning, Pandas, Scikit-learn",5 years of professional experience in financia...,0,None,
4,Data Scientist,Benjamin Davis,candidate_17.txt,48,10,"Python, SQL","Machine Learning, Pandas, Scikit-learn, Statis...","SQL, Python, Spark, AWS, Databricks, ETL, Data...",0,None,
5,Data Scientist,Sofia Garcia,candidate_11.txt,45,5,"Python, SQL","Machine Learning, Pandas, Scikit-learn, Statis...","SQL, Power BI, Tableau, Excel, Python, Data An...",0,None,
6,Data Scientist,Daniel Brown,candidate_10.txt,42,5,SQL,"Machine Learning, Pandas, Python, Scikit-learn...","B.S. in Computer Science, University of Texas ...",0,None,
7,Data Engineer,Ethan Johnson,candidate_06.txt,49,7,"AWS, Python","Airflow, Databricks, ETL, SQL, Spark",7 years of professional experience in cloud en...,0,None,
8,Data Engineer,Divya Menon,candidate_28.txt,49,8,"AWS, Python, SQL","Airflow, Databricks, ETL, Spark","SQL, Python, PostgreSQL, AWS, Linux, Data Gove...",0,None,
9,Data Engineer,Jack Thompson,candidate_29.txt,48,5,"AWS, Python","Airflow, Databricks, ETL, SQL, Spark",5 years of professional experience in devops e...,0,None,


In [55]:

review_candidate("Data Scientist", "candidate_25.txt")


JOB ROLE: Data Scientist
CANDIDATE: Samuel Clark
SCORE: 55
CURRENT LABEL: 0

JOB DESCRIPTION:

Data Scientist
We are looking for a Data Scientist with 5+ years of experience. Must have Python, SQL, machine learning, statistics, Pandas, and Scikit-learn. Experience building predictive models, running experiments, and communicating findings to product stakeholders is important. Cloud deployment experience is a plus.


MATCHED SKILLS:
['Machine Learning', 'Python', 'SQL']

MISSING DETECTED SKILLS:
['Pandas', 'Scikit-learn', 'Statistics']

FULL RESUME:

Samuel Clark
Research Data Engineer

SUMMARY
Prepared research datasets, implemented reproducible ETL, and supported model experiments.

SKILLS
Python, SQL, Spark, Machine Learning, AWS, Airflow

EXPERIENCE
5 years of professional experience in research data engineer responsibilities. Collaborated with cross-functional teams, documented technical decisions, and delivered measurable improvements to internal workflows.

EDUCATION
M.S. in Data

In [56]:

# Create a review checklist for all candidates currently labeled 0

review_checklist = needs_review[
    [
        "Job Role",
        "Candidate",
        "Resume File",
        "Score",
        "Matched Skills",
        "Missing Skills",
    ]
].copy()

review_checklist["Verified Label"] = ""
review_checklist["Reason for Decision"] = ""

display(review_checklist.reset_index(drop=True))

# Save a copy so you can continue reviewing later
review_checklist.to_csv(
    "manual_label_review.csv",
    index=False
)

print("Saved review checklist to manual_label_review.csv")


,Job Role,Candidate,Resume File,Score,Matched Skills,Missing Skills,Verified Label,Reason for Decision
0,Data Scientist,Samuel Clark,candidate_25.txt,55,"Machine Learning, Python, SQL","Pandas, Scikit-learn, Statistics",,
1,Data Scientist,Fatima Khan,candidate_07.txt,49,"Python, SQL","Machine Learning, Pandas, Scikit-learn, Statis...",,
2,Data Scientist,Divya Menon,candidate_28.txt,49,"Python, SQL","Machine Learning, Pandas, Scikit-learn, Statis...",,
3,Data Scientist,Neha Kulkarni,candidate_24.txt,49,"Python, SQL, Statistics","Machine Learning, Pandas, Scikit-learn",,
4,Data Scientist,Benjamin Davis,candidate_17.txt,48,"Python, SQL","Machine Learning, Pandas, Scikit-learn, Statis...",,
5,Data Scientist,Sofia Garcia,candidate_11.txt,45,"Python, SQL","Machine Learning, Pandas, Scikit-learn, Statis...",,
6,Data Scientist,Daniel Brown,candidate_10.txt,42,SQL,"Machine Learning, Pandas, Python, Scikit-learn...",,
7,Data Engineer,Ethan Johnson,candidate_06.txt,49,"AWS, Python","Airflow, Databricks, ETL, SQL, Spark",,
8,Data Engineer,Divya Menon,candidate_28.txt,49,"AWS, Python, SQL","Airflow, Databricks, ETL, Spark",,
9,Data Engineer,Jack Thompson,candidate_29.txt,48,"AWS, Python","Airflow, Databricks, ETL, SQL, Spark",,


Saved review checklist to manual_label_review.csv


In [57]:
review_candidate("Data Scientist", "candidate_07.txt")

JOB ROLE: Data Scientist
CANDIDATE: Fatima Khan
SCORE: 49
CURRENT LABEL: 0

JOB DESCRIPTION:

Data Scientist
We are looking for a Data Scientist with 5+ years of experience. Must have Python, SQL, machine learning, statistics, Pandas, and Scikit-learn. Experience building predictive models, running experiments, and communicating findings to product stakeholders is important. Cloud deployment experience is a plus.


MATCHED SKILLS:
['Python', 'SQL']

MISSING DETECTED SKILLS:
['Machine Learning', 'Pandas', 'Scikit-learn', 'Statistics']

FULL RESUME:

Fatima Khan
Data Engineer

SUMMARY
Built scalable ETL pipelines, improved data quality checks, and orchestrated scheduled data workflows.

SKILLS
Python, SQL, Spark, Airflow, ETL, Databricks, AWS

EXPERIENCE
6 years of professional experience in data engineer responsibilities. Collaborated with cross-functional teams, documented technical decisions, and delivered measurable improvements to internal workflows.

EDUCATION
M.S. in Data Engineer

In [58]:
review_candidate("Data Scientist", "candidate_28.txt")

JOB ROLE: Data Scientist
CANDIDATE: Divya Menon
SCORE: 49
CURRENT LABEL: 0

JOB DESCRIPTION:

Data Scientist
We are looking for a Data Scientist with 5+ years of experience. Must have Python, SQL, machine learning, statistics, Pandas, and Scikit-learn. Experience building predictive models, running experiments, and communicating findings to product stakeholders is important. Cloud deployment experience is a plus.


MATCHED SKILLS:
['Python', 'SQL']

MISSING DETECTED SKILLS:
['Machine Learning', 'Pandas', 'Scikit-learn', 'Statistics']

FULL RESUME:

Divya Menon
Database Administrator

SUMMARY
Optimized database performance, managed backups, and implemented access-control and recovery procedures.

SKILLS
SQL, Python, PostgreSQL, AWS, Linux, Data Governance

EXPERIENCE
8 years of professional experience in database administrator responsibilities. Collaborated with cross-functional teams, documented technical decisions, and delivered measurable improvements to internal workflows.

EDUCATIO

In [59]:
review_candidate("Data Scientist", "candidate_24.txt")

JOB ROLE: Data Scientist
CANDIDATE: Neha Kulkarni
SCORE: 49
CURRENT LABEL: 0

JOB DESCRIPTION:

Data Scientist
We are looking for a Data Scientist with 5+ years of experience. Must have Python, SQL, machine learning, statistics, Pandas, and Scikit-learn. Experience building predictive models, running experiments, and communicating findings to product stakeholders is important. Cloud deployment experience is a plus.


MATCHED SKILLS:
['Python', 'SQL', 'Statistics']

MISSING DETECTED SKILLS:
['Machine Learning', 'Pandas', 'Scikit-learn']

FULL RESUME:

Neha Kulkarni
Financial Data Analyst

SUMMARY
Automated financial reports, reconciled datasets, and built dashboards for forecasting and variance analysis.

SKILLS
Python, SQL, Excel, Power BI, Statistics

EXPERIENCE
5 years of professional experience in financial data analyst responsibilities. Collaborated with cross-functional teams, documented technical decisions, and delivered measurable improvements to internal workflows.

EDUCATION
M

In [60]:
review_candidate("Data Scientist", "candidate_17.txt")

JOB ROLE: Data Scientist
CANDIDATE: Benjamin Davis
SCORE: 48
CURRENT LABEL: 0

JOB DESCRIPTION:

Data Scientist
We are looking for a Data Scientist with 5+ years of experience. Must have Python, SQL, machine learning, statistics, Pandas, and Scikit-learn. Experience building predictive models, running experiments, and communicating findings to product stakeholders is important. Cloud deployment experience is a plus.


MATCHED SKILLS:
['Python', 'SQL']

MISSING DETECTED SKILLS:
['Machine Learning', 'Pandas', 'Scikit-learn', 'Statistics']

FULL RESUME:

Benjamin Davis
Data Architect

SUMMARY
Designed enterprise data platforms, governance standards, and reliable lakehouse pipelines.

SKILLS
SQL, Python, Spark, AWS, Databricks, ETL, Data Governance

EXPERIENCE
10 years of professional experience in data architect responsibilities. Collaborated with cross-functional teams, documented technical decisions, and delivered measurable improvements to internal workflows.

EDUCATION
M.S. in Informa

In [61]:
review_candidate("Data Scientist", "candidate_11.txt")

JOB ROLE: Data Scientist
CANDIDATE: Sofia Garcia
SCORE: 45
CURRENT LABEL: 0

JOB DESCRIPTION:

Data Scientist
We are looking for a Data Scientist with 5+ years of experience. Must have Python, SQL, machine learning, statistics, Pandas, and Scikit-learn. Experience building predictive models, running experiments, and communicating findings to product stakeholders is important. Cloud deployment experience is a plus.


MATCHED SKILLS:
['Python', 'SQL']

MISSING DETECTED SKILLS:
['Machine Learning', 'Pandas', 'Scikit-learn', 'Statistics']

FULL RESUME:

Sofia Garcia
Business Intelligence Analyst

SUMMARY
Translated stakeholder questions into dashboards, KPI definitions, and analytical reports.

SKILLS
SQL, Power BI, Tableau, Excel, Python, Data Analysis

EXPERIENCE
5 years of professional experience in business intelligence analyst responsibilities. Collaborated with cross-functional teams, documented technical decisions, and delivered measurable improvements to internal workflows.

EDUCAT

In [62]:
review_candidate("Data Scientist", "candidate_10.txt")


JOB ROLE: Data Scientist
CANDIDATE: Daniel Brown
SCORE: 42
CURRENT LABEL: 0

JOB DESCRIPTION:

Data Scientist
We are looking for a Data Scientist with 5+ years of experience. Must have Python, SQL, machine learning, statistics, Pandas, and Scikit-learn. Experience building predictive models, running experiments, and communicating findings to product stakeholders is important. Cloud deployment experience is a plus.


MATCHED SKILLS:
['SQL']

MISSING DETECTED SKILLS:
['Machine Learning', 'Pandas', 'Python', 'Scikit-learn', 'Statistics']

FULL RESUME:

Daniel Brown
Software Engineer

SUMMARY
Built backend services, optimized database queries, and contributed to code reviews and test automation.

SKILLS
Java, SQL, Docker, Git, AWS, Spring

EXPERIENCE
5 years of professional experience in software engineer responsibilities. Collaborated with cross-functional teams, documented technical decisions, and delivered measurable improvements to internal workflows.

EDUCATION
B.S. in Computer Scienc

In [63]:
review_candidate("Data Engineer", "candidate_06.txt")

JOB ROLE: Data Engineer
CANDIDATE: Ethan Johnson
SCORE: 49
CURRENT LABEL: 0

JOB DESCRIPTION:

Data Engineer
Hiring a Data Engineer with 5+ years of experience. Required: Python, SQL, ETL pipelines, Spark, and workflow orchestration such as Airflow. Experience with AWS or Databricks, data quality, scalable data processing, and reliable production pipelines is valuable.


MATCHED SKILLS:
['AWS', 'Python']

MISSING DETECTED SKILLS:
['Airflow', 'Databricks', 'ETL', 'SQL', 'Spark']

FULL RESUME:

Ethan Johnson
Cloud Engineer

SUMMARY
Managed cloud infrastructure, container orchestration, observability, and deployment automation.

SKILLS
AWS, Azure, Docker, Kubernetes, Terraform, Python, Linux

EXPERIENCE
7 years of professional experience in cloud engineer responsibilities. Collaborated with cross-functional teams, documented technical decisions, and delivered measurable improvements to internal workflows.

EDUCATION
B.S. in Information Systems, Arizona State University

PROJECTS
Completed

In [64]:
review_candidate("Data Engineer", "candidate_29.txt")

JOB ROLE: Data Engineer
CANDIDATE: Jack Thompson
SCORE: 48
CURRENT LABEL: 0

JOB DESCRIPTION:

Data Engineer
Hiring a Data Engineer with 5+ years of experience. Required: Python, SQL, ETL pipelines, Spark, and workflow orchestration such as Airflow. Experience with AWS or Databricks, data quality, scalable data processing, and reliable production pipelines is valuable.


MATCHED SKILLS:
['AWS', 'Python']

MISSING DETECTED SKILLS:
['Airflow', 'Databricks', 'ETL', 'SQL', 'Spark']

FULL RESUME:

Jack Thompson
DevOps Engineer

SUMMARY
Automated infrastructure provisioning and deployment workflows and improved service uptime.

SKILLS
Python, AWS, Docker, Kubernetes, Terraform, Git, Linux

EXPERIENCE
5 years of professional experience in devops engineer responsibilities. Collaborated with cross-functional teams, documented technical decisions, and delivered measurable improvements to internal workflows.

EDUCATION
B.S. in Information Technology, University of Colorado

PROJECTS
Completed pra

In [69]:
from pathlib import Path
import json
import pandas as pd

result_files = {
    "Data Scientist": "results.json",
    "Data Engineer": "data_engineer_results.json",
    "BI Analyst": "bi_results.json",
    "ML Engineer": "ml_results.json",
    "RAG Engineer": "rag_results.json",
}

rows = []

for role, filename in result_files.items():
    with open(filename, "r", encoding="utf-8") as f:
        data = json.load(f)

    ranked = data["top_matches"][:5]
    labels = ground_truth[role]

    relevant = sum(
        labels.get(Path(c["resume_path"]).name, 0)
        for c in ranked
    )

    precision = relevant / len(ranked) if ranked else 0

    rows.append({
        "Job Role": role,
        "Relevant in Top 5": f"{relevant}/{len(ranked)}",
        "Precision@5 (%)": round(precision * 100, 1),
    })

precision5_df = pd.DataFrame(rows)
display(precision5_df)

print(
    "Average Precision@5:",
    round(precision5_df["Precision@5 (%)"].mean(), 1),
    "%"
)

,Job Role,Relevant in Top 5,Precision@5 (%)
0,Data Scientist,2/5,40.0
1,Data Engineer,3/5,60.0
2,BI Analyst,4/5,80.0
3,ML Engineer,3/5,60.0
4,RAG Engineer,2/5,40.0


Average Precision@5: 56.0 %


In [70]:
from pathlib import Path

source = Path("job_matcher.py").read_text(encoding="utf-8")

keywords = [
    "match_score",
    "must_have_pass",
    "matched_skills",
    "semantic",
    "top_matches",
]

lines = source.splitlines()

for i, line in enumerate(lines):
    if any(word in line.lower() for word in keywords):
        start = max(0, i - 5)
        end = min(len(lines), i + 12)

        print(f"\n--- Lines {start + 1}-{end} ---")
        print("\n".join(
            f"{j + 1}: {lines[j]}"
            for j in range(start, end)
        ))


--- Lines 80-96 ---
80:                 "skills": set(),
81:                 "experience_years": 0,
82:                 "education": meta.get("education", "Not specified"),
83:                 "sections": [],
84:                 "excerpts": [],
85:                 "semantic_scores": [],
86:                 "full_skill_text": meta.get("skills", ""),
87:             }
88:         c = candidates[key]
89:         c["skills"].update(s.strip() for s in meta.get("skills", "").split(",") if s.strip())
90:         c["experience_years"] = max(c["experience_years"], int(meta.get("experience_years", 0) or 0))
91:         c["semantic_scores"].append(max(0.0, min(1.0, 1.0 - float(distance))))
92:         section = meta.get("section", "General")
93:         if section not in c["sections"]:
94:             c["sections"].append(section)
95:         if doc and doc not in c["excerpts"] and len(c["excerpts"]) < 4:
96:             c["excerpts"].append(doc[:500])

--- Lines 86-102 ---
86:                 "

In [72]:

import json
from pathlib import Path

result_files = {
    "Data Scientist": "results.json",
    "Data Engineer": "data_engineer_results.json",
    "BI Analyst": "bi_results.json",
    "ML Engineer": "ml_results.json",
    "RAG Engineer": "rag_results.json",
}

for role, filename in result_files.items():
    path = Path(filename)

    if not path.exists():
        print(f"{role}: File not found: {path.resolve()}")
        continue

    data = json.loads(path.read_text(encoding="utf-8"))

    print(f"\n{role}")
    for rank, candidate in enumerate(data.get("top_matches", [])[:5], 1):
        print(
            f"{rank}. {candidate.get('candidate_name')} | "
            f"{Path(candidate.get('resume_path', '')).name} | "
            f"Score: {candidate.get('match_score')}"
        )



Data Scientist
1. Aarav Sharma | candidate_01.txt | Score: 70
2. Emma Wilson | candidate_15.txt | Score: 63
3. Samuel Clark | candidate_25.txt | Score: 55
4. Fatima Khan | candidate_07.txt | Score: 49
5. Divya Menon | candidate_28.txt | Score: 49

Data Engineer
1. Fatima Khan | candidate_07.txt | Score: 72
2. Samuel Clark | candidate_25.txt | Score: 66
3. Benjamin Davis | candidate_17.txt | Score: 66
4. Ethan Johnson | candidate_06.txt | Score: 49
5. Divya Menon | candidate_28.txt | Score: 49

BI Analyst
1. Priya Reddy | candidate_04.txt | Score: 69
2. Sofia Garcia | candidate_11.txt | Score: 68
3. Sana Qureshi | candidate_30.txt | Score: 63
4. Neha Kulkarni | candidate_24.txt | Score: 62
5. Emma Wilson | candidate_15.txt | Score: 52

ML Engineer
1. Maya Patel | candidate_02.txt | Score: 61
2. Ethan Johnson | candidate_06.txt | Score: 54
3. Samuel Clark | candidate_25.txt | Score: 52
4. Ananya Iyer | candidate_09.txt | Score: 51
5. Rohan Desai | candidate_12.txt | Score: 49

RAG Engin